# Validação 02 — Plano de busca científica

## Goal

Comprovar que uma entrada validada pode ser transformada em um plano com até três consultas normalizadas e sem duplicatas, preservando a alegação e o artigo opcional.

## Setup

O notebook usa os módulos reais em `src/fatofake`. Um planejador controlado fornece consultas para que a validação seja determinística e não dependa de uma IA ou API externa nesta etapa.

In [1]:
from dataclasses import asdict
from pathlib import Path
from pprint import pprint
import sys

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "src"))

from fatofake import (
    SearchPreparationError,
    prepare_search_plan,
    validate_analysis_input,
)

## Steps

Primeiro validamos a entrada. Em seguida, o planejador fornece consultas em inglês e o módulo constrói o plano que futuramente será enviado aos conectores científicos.

In [2]:
class ControlledQueryPlanner:
    def generate_queries(self, claim: str) -> list[str]:
        return [
            "  coffee   cancer risk  ",
            "COFFEE CANCER RISK",
            "coffee neoplasms systematic review",
        ]

In [3]:
analysis_input = validate_analysis_input(
    "Tomar café aumenta o risco de câncer.",
    "https://doi.org/10.1000/xyz123",
)
search_plan = prepare_search_plan(analysis_input, ControlledQueryPlanner())
pprint(asdict(search_plan))

{'article_reference': '10.1000/xyz123',
 'claim': 'Tomar café aumenta o risco de câncer.',
 'queries': ('coffee cancer risk', 'coffee neoplasms systematic review')}


## Checks

Verificamos que o contexto foi preservado, os espaços foram normalizados, a consulta repetida foi removida e planos sem consultas são rejeitados.

In [4]:
assert search_plan.claim == analysis_input.claim
assert search_plan.article_reference == "10.1000/xyz123"
assert search_plan.queries == (
    "coffee cancer risk",
    "coffee neoplasms systematic review",
)
print("Plano válido: normalizado e sem duplicatas")

Plano válido: normalizado e sem duplicatas


In [5]:
class EmptyQueryPlanner:
    def generate_queries(self, claim: str) -> list[str]:
        return ["", "   "]

try:
    prepare_search_plan(analysis_input, EmptyQueryPlanner())
except SearchPreparationError as error:
    rejection_message = str(error)
else:
    raise AssertionError("Um plano vazio deveria ser rejeitado.")

print(rejection_message)

O planejador não produziu nenhuma consulta válida.


## Next Steps

O contrato do plano de busca estará validado quando todas as células forem executadas sem erros. A próxima etapa será consumir essas consultas no PubMed e normalizar os artigos encontrados.